<h3>Update: 3rd July, 2019


I have played around with the PyGithub package, and it has some use ability.
I am now using the json package, and have found that, given a repo ID, it can give me a full lisy of contributors to this repo, and also find the repos owned by these contributors. Next step, create a script, which
1. picks up a repo ID from a text file
2. Makes a list of the owner and contributors to this repo.
3. Finds the license of each of these repos, if the github API lists these values.
4. Sends a pre defined Pull request containing a license text file to the owner of the repo ID which was read in step 1.
5. Repeats step 3, and concatenates each result into a single file.


In [1]:
import sys
import pandas as pd
import urllib.parse
from github import Github
import json
import requests
#!{sys.executable} -m pip install PyGithub

In [3]:
#vital note
#ensure you pick projects from libraries.io database. This is important, since it will do away with lots of noisy repos
#plus I can then figure out which ecosystem a given samppled project sits in, and so what are the counterfactuals for our sample.


<h3>First I have codes pertaining to the Pygithub package.

In [4]:


# First create a Github instance:

# using username and password
g = Github("swagatam11", "***password goes here***")
sample=pd.read_csv('sample_df_RCT.csv')
#sample.iloc[5].project_id
repo = g.get_repo(19623908)
repo.url


<h3>PR code starts from below:

When the syntax says
POST /repos/:owner/:repo/pulls

what it means is that the URL I feed into my program should looks like
x=requests.post('https://api.github.com/repos/swagatam11/Robot-Judge_PS/pulls') (note that first I write "repos", then name of owner, then name of repo, then the name of the end point pulls).


In [75]:
#the following fork url must be automatically created from 
fork_url = "https://api.github.com/repos/ellliottt/robot_judge_2019/forks" # this is the URL of the repo that I am trying to create a pull request to
#https://api.github.com/repos/uweschmitt/emzed/forks
# "POST /repos/:owner/:repo/forks"
token = "28101fe940aeb964977ceb161d8dde3247e1e6ea" #my access token
headers = {"Authorization": "token %s" % token, 'Content-Type': 'application/json'}

r = requests.post(fork_url, headers=headers, verify=False) # this forks the repo in the fork_url, under my user name.
#So now I have a branch in my username, which is a copy of the repo being forked

C:\APPS\Anaconda3\lib\site-packages\urllib3\connectionpool.py:847: InsecureRequestWarning: Unverified HTTPS request is being made. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#ssl-warnings
  InsecureRequestWarning)


In [76]:
r.json

<bound method Response.json of <Response [202]>>

In [77]:
# get sha from HEAD of master branch
#this master branch is the copy of the target repo, which lives under my user name
r = requests.get("https://api.github.com/repos/swagatam11/robot_judge_2019/git/refs/heads", headers=headers)#this is the forked repo, inside my username
(r.json())
sha_master=r.json()[0]["object"]["sha"]#this is the sha of the master branch, to which i will add the license
#sha_master

In [78]:
# branch away from master branch
#the data object contains the title of the pull request which gets added to the master branch
data = {
   "ref": "refs/heads/adding_license_file",
   "sha": sha_master, # sha of master branch
 }
r = requests.post("https://api.github.com/repos/swagatam11/robot_judge_2019/git/refs", headers=headers, data=json.dumps(data))

In [79]:
#here we place the required "license file" in a variable license
import base64

apache_license = base64.b64encode(b"""
Copyright [2019] [name of copyright owner]

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.""")

license = str(apache_license, "utf-8")

In [82]:
#PUT /repos/:owner/:repo/contents/:path

# upload new license file

data = {
  "message": "added Apache 2.0 license",
  "committer": {
    "name": "Swagatam Sinha",  # fix this
    "email": "swagatam.sinha11@gmail.com"
  },
  "content": license, #this is the content file defined in the last cell
  "branch": "adding_license_file"# this branch name is the same as the reference name when you created the fork with sha_master
}

#the following line adds the license file to the new branch, created from the master branch. 
#Note, the text file name should be the name of the string which ontains the actual license text, not the variable after applying str() to it
r = requests.put("https://api.github.com/repos/swagatam11/robot_judge_2019/contents/apache_license.txt", headers=headers, data=json.dumps(data))

In [83]:
r.json()

{'commit': {'author': {'date': '2019-07-23T14:57:13Z',
   'email': 'swagatam.sinha11@gmail.com',
   'name': 'Swagatam Sinha'},
  'committer': {'date': '2019-07-23T14:57:13Z',
   'email': 'swagatam.sinha11@gmail.com',
   'name': 'Swagatam Sinha'},
  'html_url': 'https://github.com/swagatam11/robot_judge_2019/commit/08cc29de447d0c43ebd63a70e2ef759ba6fda3f7',
  'message': 'added Apache 2.0 license',
  'node_id': 'MDY6Q29tbWl0MTk4NDUxNTkyOjA4Y2MyOWRlNDQ3ZDBjNDNlYmQ2M2E3MGUyZWY3NTliYTZmZGEzZjc=',
  'parents': [{'html_url': 'https://github.com/swagatam11/robot_judge_2019/commit/8e50607719a2da151c5040d63043698b3c148e90',
    'sha': '8e50607719a2da151c5040d63043698b3c148e90',
    'url': 'https://api.github.com/repos/swagatam11/robot_judge_2019/git/commits/8e50607719a2da151c5040d63043698b3c148e90'}],
  'sha': '08cc29de447d0c43ebd63a70e2ef759ba6fda3f7',
  'tree': {'sha': 'ac899a00439c9c2b9872f16f8ea552d1ec2811eb',
   'url': 'https://api.github.com/repos/swagatam11/robot_judge_2019/git/trees/ac89

In [86]:
# create pull request in original repository we forked from:

my_url='https://api.github.com/repos/ellliottt/robot_judge_2019/pulls'

x={
    "title":"License addition request",
    "body":"Hello, We are a group of researchers at the Center for Law and Economics (ETH Zurich) studying open source software. You are receiving this email because you were randomly selected from the population of active Github repository in <name of ecosystem>, that have not adopted a license. From our studies, we would like to advise you to add the apache 2.0 license to your repository. The license text is added as a file, please add your name to it. We make this request purely from an academic research standpoint, and as such you are not under ay obligation to accept our suggestion. with Best regards, Elliott Ash.",
    "head":"swagatam11:adding_license_file",#format --> username:branch name defined above
    "base":"master"
}

token = "28101fe940aeb964977ceb161d8dde3247e1e6ea"
headers = {"Authorization": "token %s" % token, 'Content-Type': 'application/json'}

r = requests.post(my_url, headers=headers, data=json.dumps(x), verify=False)



C:\APPS\Anaconda3\lib\site-packages\urllib3\connectionpool.py:847: InsecureRequestWarning: Unverified HTTPS request is being made. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#ssl-warnings
  InsecureRequestWarning)


In [87]:
r.json()

{'_links': {'comments': {'href': 'https://api.github.com/repos/ellliottt/robot_judge_2019/issues/1/comments'},
  'commits': {'href': 'https://api.github.com/repos/ellliottt/robot_judge_2019/pulls/1/commits'},
  'html': {'href': 'https://github.com/ellliottt/robot_judge_2019/pull/1'},
  'issue': {'href': 'https://api.github.com/repos/ellliottt/robot_judge_2019/issues/1'},
  'review_comment': {'href': 'https://api.github.com/repos/ellliottt/robot_judge_2019/pulls/comments{/number}'},
  'review_comments': {'href': 'https://api.github.com/repos/ellliottt/robot_judge_2019/pulls/1/comments'},
  'self': {'href': 'https://api.github.com/repos/ellliottt/robot_judge_2019/pulls/1'},
  'statuses': {'href': 'https://api.github.com/repos/ellliottt/robot_judge_2019/statuses/08cc29de447d0c43ebd63a70e2ef759ba6fda3f7'}},
 'additions': 14,
 'assignee': None,
 'assignees': [],
 'author_association': 'NONE',
 'base': {'label': 'ellliottt:master',
  'ref': 'master',
  'repo': {'archive_url': 'https://api.gi